# English → Dutch (en-nl) Transformer — Colab Train
Port of `hkproj/pytorch-transformer` (`model.py:211` Transformer). Config already set to `en-nl` in `config.py:11` (Dutch = `nl`). 38,652 ex ~ same as Italian 32k → ~3.5h/20ep on T4 vs 12h for `en-fr 127k`.
Pipeline is ready: `model.py` + `dataset.py:5 BilingualDataset` + `train.py:184 train_model` + `translate.py:10 translate`.

In [ ]:
# 0 — Check GPU
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
!nvidia-smi -L

In [ ]:
%%capture
!pip install -q datasets tokenizers torchmetrics tensorboard

In [ ]:
# 1 — Mount Drive for checkpoints (so you don't retrain each time)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/neural-network-projects/pytorch-transformer_weights
!mkdir -p /content/drive/MyDrive/neural-network-projects/pytorch-transformer_runs

In [ ]:
# 2 — Get YOUR repo (already has en-nl config)
# If you opened this notebook from the repo itself, just cd. Otherwise clone:
import os, pathlib
if not pathlib.Path("/content/neural-network-projects").exists():
    !git clone https://github.com/abhijitdalal26/neural-network-projects.git /content/neural-network-projects
%cd /content/neural-network-projects/pytorch-transformer
!ls -lh model.py dataset.py config.py train.py translate.py
!cat config.py

In [ ]:
# 3 — Verify en-nl (English → Dutch) — similar size to Italian, good learning choice
from config import get_config
cfg = get_config()
print(cfg)
assert cfg['lang_src']=='en' and cfg['lang_tgt']=='nl', "config.py should be en-nl"
assert cfg['datasource']=='opus_books'
# Optional: override Drive paths so weights survive runtime
cfg['model_folder'] = "/content/drive/MyDrive/neural-network-projects/pytorch-transformer_weights"
cfg['tokenizer_file'] = "/content/drive/MyDrive/neural-network-projects/pytorch-transformer_weights/tokenizer_{0}.json"
cfg['experiment_name'] = "/content/drive/MyDrive/neural-network-projects/pytorch-transformer_runs/tmodel_en_nl"
cfg['num_epochs'] = 20  # set 3-5 for quick test, 20 for full (~3.5h on T4)
cfg['batch_size'] = 8
cfg['preload'] = 'latest'  # resumes if interrupted
print("Effective cfg:", cfg)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/neural-network-projects/pytorch-transformer_runs --port 6006

In [ ]:
# 4 — Train (~3.5h for 20ep on T4; 0.6h/epoch for en-nl 38k -> 14k steps = 285k for en-fr)
from train import train_model
train_model(cfg)

In [ ]:
# 5 — Inference (greedy_decode train.py:25) — <1s/sentence
from translate import translate
# monkey-patch translate to use Drive cfg if needed
import config as cfgmod
orig_get_config = cfgmod.get_config
cfgmod.get_config = lambda: cfg
print(translate("Hello, how are you?"))
print(translate("I love learning languages."))
print(translate(42))  # 42th test example: shows SOURCE/TARGET/PREDICTED
cfgmod.get_config = orig_get_config

In [ ]:
# 6 — Validation + BLEU/WER quick check (train.py:56 run_validation)
import torch
from train import get_ds, get_model
from config import get_weights_file_path, latest_weights_file_path
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_dataloader, val_dataloader, tokenizer_src, tokenizer_tgt = get_ds(cfg)
model = get_model(cfg, tokenizer_src.get_vocab_size(), tokenizer_tgt.get_vocab_size()).to(device)
ckpt = latest_weights_file_path(cfg)
print("ckpt:", ckpt)
state = torch.load(ckpt, map_location=device)
model.load_state_dict(state['model_state_dict'])
from train import run_validation
run_validation(model, val_dataloader, tokenizer_src, tokenizer_tgt, cfg['seq_len'], device, lambda m: print(m), state['global_step'], None, num_examples=5)

### Notes
- **Save vs retrain:** `train.py:262` saves every epoch `tmodel_XX.pt` (~350MB with optimizer) to Drive `model_folder` — keep them, `preload:latest` resumes. Without Drive, Colab discards runtime → retrain 3.5h again.
- **Why en-nl fast:** `opus_books en-nl 38,652 ex` vs `en-it 32,332` vs `en-fr 127,085` (4×). `d_model512 N6 90M` same per-step 0.15s, but steps/epoch 3.6k vs 14k.
- **Other good learning langs:** Full names: Catalan (ca), German (de), Greek (el), English (en), Esperanto (eo), Spanish (es 93k), Finnish (fi), French (fr), Hungarian (hu 137k), Italian (it), Dutch (nl), Norwegian (no), Polish (pl), Portuguese (pt 1.4k tiny), Russian (ru 17k Cyrillic), Swedish (sv). Try `en-es` or `en-ru` next.
- **Extra notebooks in original:** `attention_visual.ipynb` (attention heatmaps `model.py:135`), `Beam_Search.ipynb` (beam vs greedy), `Inference.ipynb` (this cell 5), `Colab_Train.ipynb`/`Local_Train.ipynb` (full loops).